# 作品タイトル：キャンパス相棒「コモ」

**氏名：三橋龍之輔**  
**第4週課題｜自分らしい、ちゃんと動くチャットボット**

勉強や大学生活でちょっと困ったとき、短い言葉で話しかけられるBotを作った。  
「勉強」「つかれた」「うれしい」などの言葉によって返事が変わり、知らない言葉にも対応する。

## 1｜Botのプロフィール

**名前：** コモ  
**役割：** 大学生活のちょっとした相談相手  
**好きなもの：** コーヒー、音楽、散歩

**自分なりの工夫：** ただ気分を返すだけでなく、勉強の話題にも対応する。また、直前に話した話題を覚えて、知らない言葉が来たときの返事を少し変える。

In [1]:
import random

bot = {
    "name": "コモ",
    "role": "大学生活の相談相手",
    "likes": ["コーヒー", "音楽", "散歩"]
}

faces = {
    "normal": "🙂", "study": "📚", "tired": "🍵", "happy": "🎉"
}

responses = {
    "study": [
        "まず10分だけ取り組んでみよう。始めることが第一歩！",
        "難しいところを一つに絞ると、進めやすいかも。",
        "{like}を少し楽しんでから、続きをやるのもいいね。"
    ],
    "tired": [
        "無理せず休憩しよう。水を飲んでみない？",
        "疲れた日は少し休むのも大切だよ。",
        "{like}で気分転換するのはどうかな？"
    ],
    "happy": [
        "それはいいね！どんなことがあったの？",
        "うれしい話を聞くと、こっちも楽しくなる！",
        "頑張った結果かもね。自分をほめよう！"
    ],
    "normal": [
        "なるほど。もう少し詳しく聞かせて！",
        "そうなんだ。今日はどんな一日だった？",
        "勉強や気分のことなら、相談に乗れるよ。"
    ]
}

print(f"{faces['normal']} {bot['name']}：こんにちは！{bot['role']}だよ。")
print("好きなもの：" + "・".join(bot['likes']))

🙂 コモ：こんにちは！大学生活の相談相手だよ。
好きなもの：コーヒー・音楽・散歩


## 2｜会話のルール

- **疲れ・否定的な言葉**を先に調べる（「元気じゃない」を「元気」と誤判定しないため）。
- 勉強の相談には勉強用、楽しい話には元気な返事を使う。
- どれにも当てはまらない入力には普通の返事をする。
- 「終了」「さようなら」「bye」なら会話を終了する。
- 同じ話題でも `random.choice()` で複数の返事から選ぶ。

In [2]:
# キーワードのリスト
study_words = ["勉強", "課題", "宿題", "テスト", "試験", "study", "homework"]
tired_words = ["元気じゃない", "疲れ", "つかれ", "眠い", "しんどい", "つらい", "tired", "not happy"]
happy_words = ["楽しい", "うれしい", "嬉しい", "元気", "最高", "happy", "great"]
end_words = ["終了", "終わり", "さようなら", "またね", "bye", "quit"]

# 同じ結果を確認しやすいよう、乱数の種を指定
rng = random.Random(4)


def answer(message, state):
    """返事・会話終了フラグを返す。stateには直前の話題を保存する。"""
    text = message.strip().lower()

    if not text:
        return "何か一言でも送ってくれるとうれしいな。", False
    elif any(word in text for word in end_words):
        return "話してくれてありがとう。またいつでも来てね！", True
    elif any(word in text for word in tired_words):
        mood = "tired"
    elif any(word in text for word in study_words):
        mood = "study"
    elif any(word in text for word in happy_words):
        mood = "happy"
    else:
        mood = "normal"

    reply = rng.choice(responses[mood]).format(like=rng.choice(bot["likes"]))
    if mood == "normal" and state["last_topic"] == "study":
        reply = "さっきの勉強の話の続きかな？ " + reply

    state["last_topic"] = mood
    return f"{faces[mood]} {reply}", False

print("会話ルールを準備できました。")

会話ルールを準備できました。


## 3｜Botと会話してみる

まず自動デモで、**勉強・疲れ・うれしい・想定外の入力・終了**を試す。実際に入力して試したい場合は、下の `demo_mode = True` を `False` に変えてセルを再実行する。`False` では「終了」と入力すると終わる。

In [3]:
demo_mode = True  # 自分で会話するときは False に変更
state = {"last_topic": None}

if demo_mode:
    messages = ["課題がなかなか終わらない", "宇宙のことを考えていた", "今日は疲れた", "今日は楽しい！", "終了"]
    for message in messages:
        print("あなた：", message)
        reply, finished = answer(message, state)
        print(f"{bot['name']}：", reply)
        if finished:
            break
else:
    print("話しかけてね（終了・byeで終わり）")
    for _ in range(10):  # 終了しなくても最大10回で止まる
        message = input("あなた：")
        reply, finished = answer(message, state)
        print(f"{bot['name']}：", reply)
        if finished:
            break
    else:
        print("コモ：今日はここまで！また話そう。")

あなた： 課題がなかなか終わらない
コモ： 📚 まず10分だけ取り組んでみよう。始めることが第一歩！
あなた： 宇宙のことを考えていた
コモ： 🙂 さっきの勉強の話の続きかな？ なるほど。もう少し詳しく聞かせて！
あなた： 今日は疲れた
コモ： 🍵 疲れた日は少し休むのも大切だよ。
あなた： 今日は楽しい！
コモ： 🎉 それはいいね！どんなことがあったの？
あなた： 終了
コモ： 話してくれてありがとう。またいつでも来てね！


## 4｜条件が正しく動くか確認する

入力の種類で気分が変わり、想定外の言葉でもエラーにならず、終了の言葉で止まるか調べる。

In [4]:
# 分類を明示的に確認（肯定語を含む否定表現に注意）
checks = [
    ("元気じゃない", "tired"),
    ("テストがある", "study"),
    ("楽しい！", "happy"),
    ("宇宙", "normal"),
]
for message, expected in checks:
    lowered = message.lower()
    if any(w in lowered for w in tired_words):
        result = "tired"
    elif any(w in lowered for w in study_words):
        result = "study"
    elif any(w in lowered for w in happy_words):
        result = "happy"
    else:
        result = "normal"
    assert result == expected
    print(f"OK：{message} → {result}")

_, finished = answer("bye", {"last_topic": None})
assert finished is True
print("OK：bye → 終了")

OK：元気じゃない → tired
OK：テストがある → study
OK：楽しい！ → happy
OK：宇宙 → normal
OK：bye → 終了


## 5｜自分のBotについて

**自分で追加したオリジナルの機能・特徴：**  
大学生活をテーマに、気分だけでなく「勉強・課題」の相談にも答えるようにした。また、「元気じゃない」が「元気」と判定されないように、疲れを表す言葉を先に調べる工夫をした。知らない言葉が来た場合も返せるようにし、直前が勉強の話題ならそれに触れる返事を追加した。

**これからどんな機能を加え、どんなロボットに育てたい？：**  
今後は会話を何回か覚えて、前に話した課題の続きを聞けるようにしたい。また、時間割や提出期限を登録し、勉強の予定を一緒に整理できるBotにしてみたい。

**動作確認：**  
「課題がなかなか終わらない」「今日は疲れた」「今日は楽しい」「宇宙」「終了」を試し、3種類以上の返事・想定外の入力・終了動作を確認した。

**AIの利用：**  
ChatGPTにPythonコードと文章の作成を手伝ってもらい、Notebookで動作を確認した。

## 6｜提出前の確認

上の **Run All（すべてを実行）** を押し、エラーがないことと会話の出力を確認してから **Ctrl + S** で保存する。提出するのはこの `.ipynb` ファイル1つ。自分で会話を試した後は、`demo_mode = True` に戻して再実行すると、提出用の会話が再現できる。